In [1]:
import torch

x = torch.ones(5)   # input tensor
y = torch.zeros(3)  # expected output
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w) + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)


In [2]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x000001652648CC70>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x000001652648C040>


# Computing Gradients
To optimize weights of parameters in the neural network, we need to compute the derivatives of our loss function with respect to parameters

In [3]:
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.0011, 0.0041, 0.2482],
        [0.0011, 0.0041, 0.2482],
        [0.0011, 0.0041, 0.2482],
        [0.0011, 0.0041, 0.2482],
        [0.0011, 0.0041, 0.2482]])
tensor([0.0011, 0.0041, 0.2482])


# Disabling Gradient Tracking

In [5]:
z = torch.matmul(x, w) + b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w) + b
print(z.requires_grad)

# another way to disable gradient tracking
z = torch.matmul(x, w) + b
z_det = z.detach()
print(z_det.requires_grad)

True
False
False


# Tensor Gradients and Jacobian Products

In [6]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

First call
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])

Second call
tensor([[8., 4., 4., 4., 4.],
        [4., 8., 4., 4., 4.],
        [4., 4., 8., 4., 4.],
        [4., 4., 4., 8., 4.]])

Call after zeroing gradients
tensor([[4., 2., 2., 2., 2.],
        [2., 4., 2., 2., 2.],
        [2., 2., 4., 2., 2.],
        [2., 2., 2., 4., 2.]])


This code demonstrates how PyTorch accumulates gradients and how you can reset them using `.zero_()`:

- `inp = torch.eye(4, 5, requires_grad=True)`: Creates a 4x5 identity matrix with gradient tracking enabled.
- `out = (inp+1).pow(2).t()`: Adds 1 to each element of `inp`, squares the result, and transposes the matrix.
- `out.backward(torch.ones_like(out), retain_graph=True)`: Computes the gradients of `out` with respect to `inp` using a tensor of ones as the gradient of the output. The `retain_graph=True` flag allows multiple backward passes on the same computation graph.
- `print(f"First call\n{inp.grad}")`: Prints the gradients after the first backward pass.
- `out.backward(torch.ones_like(out), retain_graph=True)`: Runs backward again, accumulating gradients (the gradients will be doubled compared to the first call).
- `print(f"\nSecond call\n{inp.grad}")`: Prints the accumulated gradients after the second backward pass.
- `inp.grad.zero_()`: Resets the gradients to zero.
- `out.backward(torch.ones_like(out), retain_graph=True)`: Runs backward again after zeroing the gradients.
- `print(f"\nCall after zeroing gradients\n{inp.grad}")`: Prints the gradients after reset and another backward pass.

**Summary:**
- By default, PyTorch accumulates gradients in the `.grad` attribute of tensors with `requires_grad=True`.
- You must manually zero the gradients (using `.zero_()`) before a new backward pass if you want fresh gradients.
- This behavior is important when training neural networks, as you typically want to zero gradients at the start of each optimization step to avoid unwanted accumulation.